# Extension 04 Inflatable structures

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**How does internal air pressure inflate a flat membrane into a cushion?**

```
n42 ─ n43 ─ n44 ─ n45 ─ n46 ─ n47 ─ n48
 │     │     │     │     │     │     │
n35 ─ n36 ─ n37 ─ n38 ─ n39 ─ n40 ─ n41
 │     │     │     │     │     │     │
n28 ─ n29 ─ n30 ─ n31 ─ n32 ─ n33 ─ n34
 │     │     │     │     │     │     │
n21 ─ n22 ─ n23 ─ n24 ─ n25 ─ n26 ─ n27
 │     │     │     │     │     │     │
n14 ─ n15 ─ n16 ─ n17 ─ n18 ─ n19 ─ n20
 │     │     │     │     │     │     │
n7  ─ n8  ─ n9  ─ n10 ─ n11 ─ n12 ─ n13
 │     │     │     │     │     │     │
n0  ─ n1  ─ n2  ─ n3  ─ n4  ─ n5  ─ n6
```

**Given:** the $6 \times 6$ membrane of 49 nodes $n_0 \ldots n_{48}$ with all 24 boundary nodes ${\color{#2a6fdb} \text{fixed}}$ at $z = 0\text{ m}$, uniform force density $q = 2\text{ kN/m}$, and internal air pressure $\pi = 1.0\text{ kN/m}^2$.

**Unknown:** the inflated positions ${\color{#d62828} (x_N, y_N, z_N)}$ of the 25 interior nodes.

**Equilibrium:** air pressure pushes **perpendicular to the deformed surface**, not straight up. Each triangle $t = (a, b, c)$ has area normal vector

$$ s_t = \frac{1}{2}\,\left(n_b - n_a\right) \times \left(n_c - n_a\right) $$

whose length is the triangle area and whose direction is the outward normal. One third of the pressure force $\pi\,s_t$ goes to each of its three nodes:

$$ {\color{#2e9e4a} p_i(x, y, z)} = \sum_{t \ni i} \frac{1}{3}\,\pi\,s_t $$

Because ${\color{#2e9e4a} p}$ rotates with the membrane, we update ${\color{#2e9e4a} p_x, p_y, p_z}$ from the current triangles and repeat `# equation 6.32` until the cushion settles.

<div align="center">
  <img src="https://raw.githubusercontent.com/BRG-teaching/Teaching/main/CAS/logos/fd_one_node.svg"
       style="width:100%; height:auto;"
       alt="Membrane bar tensions balancing an outward pressure load at a free node" />
  <p style="text-align:center;">
    <em>As the cushion inflates, triangle normals tilt outward so air pressure pushes both upward and sideways.</em>
  </p>
</div>

**References:**

- **JAX FDM**, https://github.com/arpastrana/jax_fdm

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">

# Network and Surface Triangles

In [1]:
import numpy as np

# nodes: 6 x 6 cells = 7 x 7 nodes n0..n48
n = 7
x, y = np.meshgrid(range(n), range(n))
x = x.flatten().astype(float)
y = y.flatten().astype(float)
z = np.zeros(n * n)

# bars and connectivity matrix C
edges = [(k, k+1) for k in range(n*n) if x[k] < n-1] + \
        [(k, k+n) for k in range(n*n) if y[k] < n-1]
C = np.zeros((len(edges), n * n))
for e, (a, b) in enumerate(edges):
    C[e, a], C[e, b] = 1, -1

# split each square cell into two triangles (alternating diagonals keep the pillow symmetric)
tris = []
for j in range(n - 1):
    for i in range(n - 1):
        v00, v10 = j * n + i, j * n + i + 1
        v01, v11 = (j + 1) * n + i, (j + 1) * n + i + 1
        if (i + j) % 2 == 0:
            tris += [(v00, v10, v11), (v00, v11, v01)]
        else:
            tris += [(v00, v10, v01), (v10, v11, v01)]

# boundary frame fixed at z = 0, interior nodes free
fixed = [k for k in range(n*n) if x[k] in (0, n-1) or y[k] in (0, n-1)]
free  = [k for k in range(n*n) if k not in fixed]
C_N, C_F = C[:, free], C[:, fixed]

# Vertical Load Versus Inflatable Follower Pressure

First solve once with the flat vertical load ($p_x = p_y = 0$). Then iterate `# equation 6.32` while recomputing ${\color{#2e9e4a} p_x, p_y, p_z}$ along the deformed triangle normals.

In [2]:
def inflatable_loads(x, y, z, pressure):
    """Nodal follower loads from internal air pressure acting normal to each triangle."""
    pts = np.column_stack([x, y, z])
    p = np.zeros_like(pts)
    for a, b, c in tris:
        s_t = 0.5 * np.cross(pts[b] - pts[a], pts[c] - pts[a])
        load_t = (pressure / 3.0) * s_t
        p[a] += load_t
        p[b] += load_t
        p[c] += load_t
    return p[:, 0], p[:, 1], p[:, 2]


q = 2.0 * np.ones(len(edges))
Q = np.diag(q)
A = C_N.T @ Q @ C_N
pressure = 1.0      # internal gauge pressure (kN/m^2)

# 1. vertical load only (one linear solve from the flat grid)
p_x0, p_y0, p_z0 = inflatable_loads(x, y, z, pressure)
x_vert, y_vert, z_vert = x.copy(), y.copy(), z.copy()
x_vert[free] = np.linalg.solve(A, p_x0[free] - C_N.T @ Q @ C_F @ x_vert[fixed])
y_vert[free] = np.linalg.solve(A, p_y0[free] - C_N.T @ Q @ C_F @ y_vert[fixed])
z_vert[free] = np.linalg.solve(A, p_z0[free] - C_N.T @ Q @ C_F @ z_vert[fixed])

# 2. inflatable follower pressure loop
x_inf, y_inf, z_inf = x.copy(), y.copy(), z.copy()
for step in range(40):
    p_x, p_y, p_z = inflatable_loads(x_inf, y_inf, z_inf, pressure)

    # equation 6.32
    x_N = np.linalg.solve(A, p_x[free] - C_N.T @ Q @ C_F @ x_inf[fixed])
    y_N = np.linalg.solve(A, p_y[free] - C_N.T @ Q @ C_F @ y_inf[fixed])
    z_N = np.linalg.solve(A, p_z[free] - C_N.T @ Q @ C_F @ z_inf[fixed])

    move = np.max(np.sqrt((x_N - x_inf[free])**2 + (y_N - y_inf[free])**2 + (z_N - z_inf[free])**2))
    x_inf[free], y_inf[free], z_inf[free] = x_N, y_N, z_N
    if move < 1e-10:
        break

swell = np.max(np.hypot(x_inf - x, y_inf - y))
print("inflation steps            =", step + 1)
print("vertical load crown z24    =", round(z_vert[24], 2), "m  lateral swell = 0.0 m")
print("inflatable crown z24       =", round(z_inf[24], 2), "m  lateral swell =", round(swell, 2), "m")
print("max horizontal air load    =", round(np.hypot(p_x[free], p_y[free]).max(), 2), "kN")

inflation steps            = 31
vertical load crown z24    = 1.32 m  lateral swell = 0.0 m
inflatable crown z24       = 1.58 m  lateral swell = 0.26 m
max horizontal air load    = 0.94 kN


# Viewer

Gray: one vertical load solve. Black: inflated cushion under normal air pressure, with green follower pressure vectors ${\color{#2e9e4a} p_i}$ perpendicular to the curved membrane.

In [3]:
from session_py import Arrowhead, Color, Line, Matrix, Session
from session_jupyter import show

# the colours of the figures: free nodes red, fixed nodes blue, loads green
RED   = Color(0.84, 0.16, 0.16, 1.0)
BLUE  = Color(0.16, 0.44, 0.86, 1.0)
GREEN = Color(0.18, 0.62, 0.29, 1.0)
GRAY  = Color(0.70, 0.70, 0.70, 1.0)

f = q * np.sqrt((C @ x_inf)**2 + (C @ y_inf)**2 + (C @ z_inf)**2)
colors = [Color(0.65 - 0.65 * fv / f.max(), 0.65 - 0.65 * fv / f.max(), 0.65 - 0.65 * fv / f.max(), 1.0) for fv in f]

session = Session("fdm_ext04_inflatable_cushion")
session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x_vert, y_vert, z_vert]).tolist()), [GRAY] * len(edges), [1.5] * len(edges))
session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x_inf, y_inf, z_inf]).tolist()), colors, [2.8] * len(edges))

for net in range(2):
    for k in range(n * n):
        pt = session.objects.points[net * n * n + k]
        pt.pointcolor = BLUE if k in fixed else RED
        pt.width = 11.0 if k in fixed else 6.0

for k in free:
    arrow = Line(x_inf[k], y_inf[k], z_inf[k], x_inf[k] + 0.5 * p_x[k], y_inf[k] + 0.5 * p_y[k], z_inf[k] + 0.5 * p_z[k])
    arrow.linecolor = GREEN
    arrow.width = 2.4
    arrow.arrowhead = Arrowhead.END
    session.add_line(arrow)

show(session, "fdm_ext04_inflatable_cushion")

# Observation

Under a vertical load $x_N$ and $y_N$ stay frozen on the flat grid. Under real air pressure the rotating normals push the sloping sides outward by $0.26\text{ m}$ and lift the crown from $1.32\text{ m}$ to $1.58\text{ m}$ while $A = C_N^{T}\,Q\,C_N$ stays constant at every step. Try raising `pressure = 1.6`: just like Exercise 01, if the pressure grows faster than $q\,l$ can restrain, no equilibrium exists and the cushion bursts.